# Notebook 1: PATSTAT High-Recall Baseline Extraction
Extracts the broad candidate corpus of drone-related patent applications from PATSTAT `PROD` using boolean combinations of CPC/IPC classes and domain keywords, runs initial quality diagnostics, and saves the raw baseline to Parquet.

In [1]:
import os
from datetime import date

import pandas as pd
from sqlalchemy import func, or_, and_, not_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons (applicants and inventors)
    TLS207_PERS_APPLN,          # person <-> application links
    TLS209_APPLN_IPC,           # IPC classification
    TLS224_APPLN_CPC,           # CPC classification
    TLS231_INPADOC_LEGAL_EVENT, # legal events
)

In [2]:
# Connect to PATSTAT PROD environment
patstat = PatstatClient(env="PROD")
db = patstat.orm()

In [3]:
# Multi-tier taxonomy and exclusion lists

# Primary Anchor Keywords (High Precision: Directly define flying UAV platforms)
DRONE_ANCHOR_KEYWORDS = [
    "%unmanned aerial vehicle%",
    "%unmanned aircraft%",
    "%remotely piloted aircraft%",
    "%RPAS%",
    "%UAV%",
]

# Secondary Functional Keywords (Dual-Use: Require co-occurrence with Core CPC/IPC or Anchor Keywords)
DRONE_SECONDARY_KEYWORDS = [
    "%drone%",
    "%quadcopter%",
    "%multirotor%",
    "%hexacopter%",
]

# Exclusion Keywords (Filter out non-flying autonomous/robotic domains)
DRONE_EXCLUSION_KEYWORDS = [
    "%toy%",
    "%underwater%",
    "%submersible%",
    "%marine%",
]


DRONE_CORE_CPC_CODES = [
    "B64U%",        # Primary CPC class dedicated to UAVs
    "B64C39/02%",   # Legacy CPC class for remote-controlled aircraft
]

DRONE_SECONDARY_CPC_CODES = [
    "G08G5/00%",    # Aeronautical traffic control & UTM
    "G05D1/00%",    # Control of position / course / flight control
]

DRONE_CORE_IPC_CODES = [
    "B64U%",
    "B64C39/02%",
]

DRONE_SECONDARY_IPC_CODES = [
    "G08G5/00%",
    "G05D1/00%",
]

In [4]:
# Build Atomic Filter Conditions
core_cpc_f = [TLS224_APPLN_CPC.cpc_class_symbol.like(c) for c in DRONE_CORE_CPC_CODES]
core_ipc_f = [TLS209_APPLN_IPC.ipc_class_symbol.like(c) for c in DRONE_CORE_IPC_CODES]

sec_cpc_f = [TLS224_APPLN_CPC.cpc_class_symbol.like(c) for c in DRONE_SECONDARY_CPC_CODES]
sec_ipc_f = [TLS209_APPLN_IPC.ipc_class_symbol.like(c) for c in DRONE_SECONDARY_IPC_CODES]

title_anchor_f = [TLS202_APPLN_TITLE.appln_title.ilike(k) for k in DRONE_ANCHOR_KEYWORDS]
abstr_anchor_f = [TLS203_APPLN_ABSTR.appln_abstract.ilike(k) for k in DRONE_ANCHOR_KEYWORDS]

title_sec_f = [TLS202_APPLN_TITLE.appln_title.ilike(k) for k in DRONE_SECONDARY_KEYWORDS]
abstr_sec_f = [TLS203_APPLN_ABSTR.appln_abstract.ilike(k) for k in DRONE_SECONDARY_KEYWORDS]

title_excl_f = [TLS202_APPLN_TITLE.appln_title.ilike(k) for k in DRONE_EXCLUSION_KEYWORDS]
abstr_excl_f = [TLS203_APPLN_ABSTR.appln_abstract.ilike(k) for k in DRONE_EXCLUSION_KEYWORDS]

In [5]:
# Restructured Boolean Logic Clauses
standalone_core = or_(
    *core_cpc_f,
    *core_ipc_f,
    *title_anchor_f,
    *abstr_anchor_f
)

# Secondary codes MUST co-occur with explicit Anchor terms
secondary_code_match = and_(
    or_(*sec_cpc_f, *sec_ipc_f),
    or_(*title_anchor_f, *abstr_anchor_f)
)

# Secondary keywords MUST co-occur with Core Codes
secondary_kw_match = and_(
    or_(*title_sec_f, *abstr_sec_f),
    or_(*core_cpc_f, *core_ipc_f)
)

# Master Conditions
inclusion_clause = or_(
    standalone_core,
    secondary_code_match,
    secondary_kw_match
)

exclusion_clause = not_(or_(*title_excl_f, *abstr_excl_f))

In [6]:
MIN_FILING_DATE = date(2000, 1, 1)
MAX_FILING_DATE = date(2023, 1, 1)

query = (
    db.query(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_nr,
        TLS201_APPLN.appln_filing_date,
        TLS201_APPLN.appln_kind,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract,
        func.string_agg(func.distinct(TLS206_PERSON.doc_std_name), ', ').label("applicants"),
        func.string_agg(func.distinct(TLS224_APPLN_CPC.cpc_class_symbol), ', ').label("cpc_codes"),
        func.string_agg(func.distinct(TLS209_APPLN_IPC.ipc_class_symbol), ', ').label("ipc_codes"),
    )
    .join(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .outerjoin(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
    .outerjoin(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
    .join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
    .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
    .filter(
        and_(
            inclusion_clause,
            exclusion_clause,
            TLS201_APPLN.appln_filing_date >= MIN_FILING_DATE,
            TLS201_APPLN.appln_filing_date < MAX_FILING_DATE
        )
    )
    .group_by(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_nr,
        TLS201_APPLN.appln_filing_date,
        TLS201_APPLN.appln_kind,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract
    )
)

### Execute Query on PATSTAT PROD
**Warning:** Running `query.all()` queries the full production database and takes several minutes. Only run when regenerating the raw baseline.

In [7]:
results = query.all()
df = pd.DataFrame(results)
print(f"Total matching patent applications retrieved: {len(df):,}")

Total matching patent applications retrieved: 184,728


In [8]:
# Baseline Data Diagnostics
print("Abstract missingness:", round(df['appln_abstract'].isna().mean(), 4))
print("Title missingness:", round(df['appln_title'].isna().mean(), 4))
print("Total Applications (appln_id):", df['appln_id'].nunique())
print("Total Patent Families (docdb_family_id):", df['docdb_family_id'].nunique())

df['filing_year'] = pd.to_datetime(df['appln_filing_date'], errors='coerce').dt.year
print("\nFiling year distribution:")
print(df['filing_year'].value_counts().sort_index())
print("\nPlaceholder/Invalid date count:", df['filing_year'].isna().sum())

df.head()

Abstract missingness: 0.0
Title missingness: 0.0
Total Applications (appln_id): 184728
Total Patent Families (docdb_family_id): 151883

Filing year distribution:
filing_year
2000      705
2001      714
2002      677
2003      595
2004      744
2005      877
2006      898
2007     1054
2008     1085
2009     1127
2010     1240
2011     1512
2012     1891
2013     2384
2014     3822
2015     8896
2016    16622
2017    23293
2018    23279
2019    22021
2020    24735
2021    23711
2022    22846
Name: count, dtype: int64

Placeholder/Invalid date count: 0


,appln_id,docdb_family_id,appln_auth,appln_nr,appln_filing_date,appln_kind,appln_title,appln_abstract,applicants,cpc_codes,ipc_codes,filing_year
0,41231070,19745082,WO,0000011,2000-02-14,W,WATER FAUCET FOR SANITARY TANK,The invention relates to a water faucet for a ...,FLORES GUTIERREZ SALVADOR,"E03D 1/32, E03D 1/186","E03D 1/18, E03D 1/32",2000
1,266718711,22364859,CO,00001054,2000-01-11,A,ESTRUCTURA ABSORBENTE DE ALTA RESISTENCIA,Una estructura absorbente que tiene una integr...,"BKI HOLDING CORP, JACEK K DUTKIEWICZ, RYAN K H...","Y10T 442/653, B32B 5/26, Y10T 442/30, Y10T 4...","B32B 27/04, D04H 1/40, A61F 13/15, B32B ...",2000
2,3447771,4074135,WO,0000129,2000-11-23,W,"SPEED CONTROLLER AND LIMITER, WITH ANTI-THEFT ...","Speed controller and limiter, with anti-theft ...","WENDLER CARLOS EDUARDO, DOS SANTOS LINEU JOSE","B60K 31/18, B60W2710/0616, B60R 25/04, B60R ...","B60R 25/10, B60R 25/04, B60K 31/04, B60K 3...",2000
3,4321269,4163333,WO,0000202,2000-02-25,W,DUAL CONTROL ANALOG DELAY ELEMENT,A delay line comprised of analog delay element...,"LEE KI-JUN, BHULLAR GURPREET, MOSAID TECHNOLOG...","H03K2005/00032, H03K 5/131, H03K 5/13, H03...","H03H 11/26, H03K 5/13",2000
4,3447983,22922702,BR,0000310,2000-02-04,A,Processo de modulação digital e modulador que ...,Patente de Invenção: <B>'PROCESSO DE MODULAçãO...,"MATRA NORTEL COMMUNICATIONS, MARQUE-PUCHEU GER...",None,"H04L 27/00, H04L 27/20",2000


In [9]:
# Save baseline corpus to Parquet
baseline_path = "flying_drones_baseline_prod.parquet"
df.to_parquet(baseline_path, index=False)

size_mb = os.path.getsize(baseline_path) / (1024 * 1024)
print(f"Successfully saved '{baseline_path}': {len(df):,} rows ({size_mb:.2f} MB)")

Successfully saved 'flying_drones_baseline_prod.parquet': 184,728 rows (102.07 MB)
